# 07. 통합 리포트 — SDF-Xplain 한눈에 보기

Phase 1~9 의 산출물을 하나의 스토리로 엮는다.

**시나리오**: 자동차 파워트레인 가공 라인의 스핀들 베어링 + CNC 공정 파라미터 → 품질/고장 원인 분석 + 개입 효과 추정.

**핵심 질문 3가지**
1. **언제** 고장이 발생할 위험이 높은가?
2. **무엇이** 그 위험을 만들었는가?
3. **어떻게** 조치하면 위험이 줄어드는가?

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import joblib

from career_kia.config import PROCESSED_DIR, PROJECT_ROOT
from career_kia.models.train import load_feature_matrix

feat = pd.read_parquet(PROCESSED_DIR / 'features.parquet')
model = joblib.load(PROJECT_ROOT / 'models_artifacts' / 'hybrid_model.joblib')
X, y, _ = load_feature_matrix()
proba = model.predict_proba(X)[:, 1]

## 1. 데이터 요약

In [ ]:
print(f'총 배치 수: {len(feat):,}')
print(f'피처 수: {X.shape[1]}')
print(f'실제 고장 비율: {y.mean()*100:.2f}%')
print(f'결함 유형 분포:')
print(feat["vibration_fault_type"].value_counts())

## 2. 질문 1 — **언제** 위험한가? (모델 성능)

In [ ]:
summary = json.loads((PROJECT_ROOT / 'models_artifacts' / 'cv_summary.json').read_text())
rows = []
for name, m in summary.items():
    rows.append({
        'model': name,
        'ROC-AUC': f"{m['roc_auc_mean']:.3f}",
        'PR-AUC': f"{m['pr_auc_mean']:.3f}",
        'F1': f"{m['f1_mean']:.3f}",
        'P@R=0.9': f"{m['precision_at_recall_0.9_mean']:.3f}",
    })
display(pd.DataFrame(rows))

## 3. 질문 2 — **무엇이** 기여했는가? (XAI)

In [ ]:
shap_global = pd.read_parquet(PROJECT_ROOT / 'xai_artifacts' / 'shap_global.parquet')
print('전역 SHAP 상위 10:')
display(shap_global.head(10))

In [ ]:
# 고위험 대표 배치의 자연어 설명 1건
narratives = json.loads((PROJECT_ROOT / 'xai_artifacts' / 'top_risk_narratives.json').read_text(encoding='utf-8'))
print(narratives[0]['narrative'])

## 4. 질문 3 — **어떻게** 줄이는가? (인과추론 개입 효과)

In [ ]:
ate = json.loads((PROJECT_ROOT / 'causal_artifacts' / 'ate_summary.json').read_text(encoding='utf-8'))
rows = []
for trt, info in ate.items():
    rows.append({
        'treatment': trt,
        '기준값': info['control_value'],
        '개입값': info['treatment_value'],
        'ATE (%p)': f"{info['ate']*100:+.2f}",
        'refute(RCC) (%p)': f"{info['refutation']['random_common_cause']*100:+.2f}",
        'refute(subset) (%p)': f"{info['refutation']['data_subset_refuter']*100:+.2f}",
    })
display(pd.DataFrame(rows))

## 5. 핵심 메시지 — 현장용 요약 3 문장

1. **고장 조기 예측** — 하이브리드 모델(LGBM+GAM)이 ROC-AUC 0.96 / PR-AUC 0.86 으로 희소 고장(3.4%)을 안정적으로 포착한다.
2. **원인 설명** — 상위 기여 변수는 토크 · BPFO 포락선 · 공구 마모로 베어링 외륜 결함 + 공정 조건이 결합된 고장이 다수. SHAP 자연어 설명이 각 고장 배치마다 생성되어 의사결정 근거를 제공한다.
3. **실행 가능한 처방** — 공구 마모 조기 교체(200→150분) · 토크 하향(60→40Nm) · 회전속도 정상화(1300→1600rpm) 각각이 고장 확률을 수 %p 감소시키는 것으로 인과 추정. Refutation 2종 통과.

→ Streamlit 대시보드로 현장 엔지니어가 실시간 검토 및 What-if 시뮬레이션 가능.